# Walmart Dual-Source O*NET and Lightcast Analysis

This notebook compares foundational Knowledge, Skills, and Abilities in fast- versus slow-growth occupations using O*NET, and current employer-requested skills using Lightcast.

**Before running:** update the paths in Section 0, then run all cells from top to bottom.


In [38]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


## 0. FILE PATHS — EDIT THESE


In [39]:
STAFFING_PATTERN_FILE = Path(r"C:\Users\301533\Documents\Walmart\2025 Staffing Pattern (2).xlsx")
PROJECTIONS_FILE = Path(r"C:\Users\301533\Documents\Walmart\ltop-04-2024to2034 (7).xlsx")

ONET_SKILLS_FILE = Path(r"C:\Users\301533\Documents\AI Analysis\AI Skills\WORKBank Outputs\ONET Docs\Skills (3).xlsx")
ONET_KNOWLEDGE_FILE = Path(r"C:\Users\301533\Documents\AI Analysis\AI Skills\WORKBank Outputs\ONET Docs\Knowledge.xlsx")
ONET_ABILITIES_FILE = Path(r"C:\Users\301533\Downloads\Abilities.xlsx")

# REPLACE LIGHTCAST_FOLDER WITH DIRECT FILE PATH:
LIGHTCAST_FILE = Path(r"G:\Shared drives\EO_OPERATIONS\Grants\PCIC Analysis\Job Posting Data\raw_data\df_2016_2025.csv")# (Or C:\Users\301533\Documents\Walmart\df_2016_2025.csv depending on where your copy is saved)

OUTPUT_FOLDER = Path(r"C:\Users\301533\Documents\Walmart_Analysis_Output\Dual_Source_Skills_Analysis")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


## 1. ANALYSIS SETTINGS


In [40]:
SECTOR_NAICS_PREFIXES = {
    "Healthcare": ["62"],
    "Manufacturing": ["31", "32", "33"],
    "Semiconductors": ["334"],
}

STAFFING_AREATYPE = "01"
STAFFING_AREA = "000004"
STAFFING_OCCCLASS = "N"

# Quartiles are more stable than selecting only 10 occupations.
HIGH_GROWTH_QUANTILE = 0.75
LOW_GROWTH_QUANTILE = 0.25
MIN_SECTOR_EMPLOYMENT = 0
TOP_N_RESULTS = 10

# O*NET: show Importance and Level separately. Do not combine categories/scales.
ONET_SCALES = {"IM": "Importance", "LV": "Level"}

# Lightcast loader modes:
# "auto" = detect raw-posting or pre-aggregated structure.
# "raw" = one posting per row, with a posting ID and a skill/list-of-skills field.
# "summary" = one SOC-skill row with a skill-posting count or demand percentage.
LIGHTCAST_MODE = "auto"

FAST_COLOR = "#17C3B2"
SLOW_COLOR = "#F79732"


## 2. UTILITIES


In [41]:
def norm(text):
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def clean_soc(value):
    if pd.isna(value):
        return np.nan
    digits = re.sub(r"\D", "", str(value))
    return f"{digits[:2]}-{digits[2:6]}" if len(digits) >= 6 else np.nan


def detailed_soc(value):
    return bool(pd.notna(value) and re.match(r"^\d{2}-\d{4}$", str(value)) and not str(value).endswith("0000"))


def clean_naics(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\.0$", "", str(value).strip())


def number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce",
    )


def find_col(columns, candidates, excludes=()):
    normalized = {c: norm(c) for c in columns}
    for candidate in candidates:
        key = norm(candidate)
        for col, col_norm in normalized.items():
            if key == col_norm and not any(norm(x) in col_norm for x in excludes):
                return col
    for candidate in candidates:
        key = norm(candidate)
        for col, col_norm in normalized.items():
            if key in col_norm and not any(norm(x) in col_norm for x in excludes):
                return col
    return None


def weighted_mean(values, weights):
    v = pd.to_numeric(values, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce")
    ok = v.notna() & w.notna() & (w > 0)
    return np.average(v[ok], weights=w[ok]) if ok.any() else v.mean()


def read_table(path, **kwargs):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False, **kwargs)
    if suffix in (".xlsx", ".xls"):
        return pd.read_excel(path, **kwargs)
    if suffix in (".parquet", ".pq"):
        return pd.read_parquet(path, **kwargs)
    raise ValueError(f"Unsupported file type: {path}")


def require_files():
    required = [STAFFING_PATTERN_FILE, PROJECTIONS_FILE, ONET_SKILLS_FILE,
                ONET_KNOWLEDGE_FILE, ONET_ABILITIES_FILE, LIGHTCAST_FILE]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required file(s):\n" + "\n".join(missing))
    if not LIGHTCAST_FOLDER.exists():
        raise FileNotFoundError(f"Lightcast folder not found: {LIGHTCAST_FOLDER}")


## 3. STAFFING PATTERN: DEFINE OCCUPATIONS WITHIN EACH SECTOR


In [42]:
def load_staffing():
    df = pd.read_excel(STAFFING_PATTERN_FILE)
    df.columns = [str(c).strip() for c in df.columns]
    area_type = find_col(df.columns, ["areatype", "area type"])
    area = find_col(df.columns, ["area"])
    industry = find_col(df.columns, ["indcode", "naics", "industry code"])
    occclass = find_col(df.columns, ["occclass", "occupation class"])
    soc = find_col(df.columns, ["occcd", "soc code", "soc"])
    employment = find_col(df.columns, ["empl", "employment", "jobs"])
    title = find_col(df.columns, ["occtitle", "occupation title", "occupation", "occ name", "occupation description"])
    required = {"area type": area_type, "area": area, "industry": industry,
                "occupation class": occclass, "SOC": soc, "employment": employment}
    absent = [k for k, v in required.items() if v is None]
    if absent:
        raise ValueError(f"Staffing file missing fields {absent}. Columns: {list(df.columns)}")

    out = pd.DataFrame({
        "AreaType": df[area_type].astype(str).str.replace(".0", "", regex=False).str.zfill(2),
        "Area": df[area].astype(str).str.replace(".0", "", regex=False).str.zfill(6),
        "NAICS": df[industry].apply(clean_naics),
        "OccClass": df[occclass].astype(str).str.strip(),
        "SOC": df[soc].apply(clean_soc),
        "Sector_Employment": number(df[employment]),
        "Staffing_Occupation": df[title].astype(str).str.strip() if title else "",
    })
    out = out[(out.AreaType == STAFFING_AREATYPE) & (out.Area == STAFFING_AREA) &
              (out.OccClass == STAFFING_OCCCLASS) & out.SOC.apply(detailed_soc)].copy()

    parts = []
    for sector, prefixes in SECTOR_NAICS_PREFIXES.items():
        s = out[out.NAICS.apply(lambda x: any(str(x).startswith(p) for p in prefixes))].copy()
        s = s.groupby("SOC", as_index=False).agg(
            Sector_Employment=("Sector_Employment", "sum"),
            Staffing_Occupation=("Staffing_Occupation", lambda x: next((v for v in x if v and v.lower() != "nan"), "")),
        )
        s = s[s.Sector_Employment >= MIN_SECTOR_EMPLOYMENT]
        s["Sector"] = sector
        parts.append(s)
    return pd.concat(parts, ignore_index=True)


## 4. PROJECTIONS: LOAD TITLES AND GROWTH, WITH TITLE FALLBACK


In [43]:
def load_projections():
    xl = pd.ExcelFile(PROJECTIONS_FILE)
    match = None
    for sheet in xl.sheet_names:
        for header in range(15):
            test = pd.read_excel(PROJECTIONS_FILE, sheet_name=sheet, header=header)
            test.columns = [str(c).strip() for c in test.columns]
            soc = find_col(test.columns, ["soc code", "soccode", "occcd", "occupation code", "soc"])
            if soc is None or test[soc].apply(clean_soc).notna().sum() < 5:
                continue
            title = find_col(test.columns, ["occupation title", "occtitle", "occupation name", "occ name",
                                                   "occupation description", "occdesc", "occupation"], excludes=["code"])
            base = find_col(test.columns, ["2024 employment", "employment 2024", "2024 estimate",
                                                  "base employment", "estimated employment"])
            projected = find_col(test.columns, ["2034 employment", "employment 2034", "2034 projection",
                                                       "projected employment", "projection employment"])
            if base and projected:
                match = (test, sheet, header, soc, title, base, projected)
                break
        if match:
            break
    if not match:
        raise ValueError("Could not detect SOC, 2024 employment, and 2034 employment columns in projections file.")

    df, sheet, header, soc, title, base, projected = match
    print(f"Projection sheet={sheet!r}, header row={header + 1}, title column={title!r}")
    out = pd.DataFrame({
        "SOC": df[soc].apply(clean_soc),
        "Projection_Occupation": df[title].astype(str).str.strip() if title else "",
        "Base_Employment": number(df[base]),
        "Projected_Employment": number(df[projected]),
    })
    out["Numeric_Change"] = out.Projected_Employment - out.Base_Employment
    out["Growth_Rate"] = np.where(out.Base_Employment > 0, out.Numeric_Change / out.Base_Employment, np.nan)
    out = out[out.SOC.apply(detailed_soc)].sort_values("Base_Employment", ascending=False).drop_duplicates("SOC")
    return out


def build_occupation_universe(staffing, projections):
    df = staffing.merge(projections, on="SOC", how="inner")
    df["Occupation"] = df.Projection_Occupation.replace({"": np.nan, "nan": np.nan}).fillna(
        df.Staffing_Occupation.replace({"": np.nan, "nan": np.nan})
    )
    missing = df.Occupation.isna().sum()
    if missing:
        print(f"WARNING: {missing} merged SOC rows still lack titles; their SOC codes will be used as labels.")
        df["Occupation"] = df.Occupation.fillna(df.SOC)
    df = df.dropna(subset=["Growth_Rate"])

    def label_groups(g):
        low = g.Growth_Rate.quantile(LOW_GROWTH_QUANTILE)
        high = g.Growth_Rate.quantile(HIGH_GROWTH_QUANTILE)
        g = g.copy()
        g["Growth_Group"] = np.select(
            [g.Growth_Rate <= low, g.Growth_Rate >= high],
            ["Slow-Growth Quartile", "Fast-Growth Quartile"], default="Middle 50%")
        return g

    return df.groupby("Sector", group_keys=False).apply(label_groups).reset_index(drop=True)


## 5. O*NET: IMPORTANCE AND LEVEL, SEPARATED BY K/S/A


In [44]:
def load_onet_file(path, ksa_type):
    df = pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]
    soc = find_col(df.columns, ["O*NET-SOC Code", "ONET SOC Code", "SOC"])
    element = find_col(df.columns, ["Element Name", "Element"])
    value = find_col(df.columns, ["Data Value", "Value", "Score"])
    scale_id = find_col(df.columns, ["Scale ID"])
    scale_name = find_col(df.columns, ["Scale Name"])
    if not all([soc, element, value]) or not (scale_id or scale_name):
        raise ValueError(f"Could not detect required O*NET columns in {path.name}: {list(df.columns)}")
    scale_raw = df[scale_id].astype(str).str.upper().str.strip() if scale_id else df[scale_name].astype(str).str.lower().str.strip()
    if scale_id:
        scale = scale_raw.map(ONET_SCALES)
    else:
        scale = scale_raw.map({"importance": "Importance", "level": "Level"})
    out = pd.DataFrame({"SOC": df[soc].apply(clean_soc), "KSA_Type": ksa_type,
                        "KSA_Name": df[element].astype(str).str.strip(), "Scale": scale,
                        "ONET_Score": number(df[value])})
    out = out.dropna(subset=["SOC", "KSA_Name", "Scale", "ONET_Score"])
    return out.groupby(["SOC", "KSA_Type", "KSA_Name", "Scale"], as_index=False).ONET_Score.mean()


def analyze_onet(universe):
    onet = pd.concat([
        load_onet_file(ONET_SKILLS_FILE, "Skill"),
        load_onet_file(ONET_KNOWLEDGE_FILE, "Knowledge"),
        load_onet_file(ONET_ABILITIES_FILE, "Ability"),
    ], ignore_index=True)
    selected = universe[universe.Growth_Group != "Middle 50%"].merge(onet, on="SOC", how="inner")
    rows = []
    keys = ["Sector", "Growth_Group", "KSA_Type", "Scale", "KSA_Name"]
    for key, g in selected.groupby(keys):
        rows.append(dict(zip(keys, key)) | {
            "Mean_Score": g.ONET_Score.mean(),
            "Employment_Weighted_Score": weighted_mean(g.ONET_Score, g.Sector_Employment),
            "Occupation_Count": g.SOC.nunique(),
        })
    summary = pd.DataFrame(rows)
    pivot = summary.pivot_table(
        index=["Sector", "KSA_Type", "Scale", "KSA_Name"], columns="Growth_Group",
        values="Employment_Weighted_Score").reset_index()
    pivot.columns.name = None
    for c in ["Fast-Growth Quartile", "Slow-Growth Quartile"]:
        if c not in pivot:
            pivot[c] = np.nan
    pivot["Fast_Minus_Slow"] = pivot["Fast-Growth Quartile"] - pivot["Slow-Growth Quartile"]
    pivot["Absolute_Difference"] = pivot.Fast_Minus_Slow.abs()

    top = summary.sort_values("Employment_Weighted_Score", ascending=False).groupby(
        ["Sector", "Growth_Group", "KSA_Type", "Scale"], group_keys=False).head(TOP_N_RESULTS)
    differentiators = pivot.sort_values("Fast_Minus_Slow", ascending=False).groupby(
        ["Sector", "KSA_Type", "Scale"], group_keys=False).head(TOP_N_RESULTS)
    return selected, summary, top, pivot, differentiators


## 6. LIGHTCAST: CURRENT EMPLOYER-REQUESTED SKILLS


In [45]:
import ast
from collections import Counter


def parse_skill_list(x):
    """Safely parse skill list strings from Lightcast CSV data."""
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s in ("", "[]", "nan", "None"):
        return []
    try:
        out = ast.literal_eval(s)
        if isinstance(out, list):
            return [str(v).strip() for v in out if str(v).strip()]
    except Exception:
        pass
    try:
        out = json.loads(s)
        if isinstance(out, list):
            return [str(v).strip() for v in out if str(v).strip()]
    except Exception:
        pass
    return []


def analyze_lightcast(universe):
    print(f"Reading Lightcast file: {LIGHTCAST_FILE.name}")

    target_socs = set(universe[universe.Growth_Group != "Middle 50%"]["SOC"])
    chunksize = 100_000
    cols_to_use = ["SOC_5", "YEAR", "SKILLS_NAME", "SOFTWARE_SKILLS_NAME"]

    soc_skill_counts = Counter()
    soc_total_postings = Counter()

    for chunk in pd.read_csv(
        LIGHTCAST_FILE,
        usecols=lambda c: c in cols_to_use,
        chunksize=chunksize,
        low_memory=False,
    ):
        if "YEAR" in chunk.columns:
            chunk["YEAR"] = pd.to_numeric(chunk["YEAR"], errors="coerce")
            chunk = chunk[chunk["YEAR"] == 2025]

        if chunk.empty:
            continue

        chunk["SOC"] = chunk["SOC_5"].apply(clean_soc)
        chunk = chunk[chunk["SOC"].isin(target_socs)]
        if chunk.empty:
            continue

        for _, row in chunk.iterrows():
            soc = row["SOC"]
            soc_total_postings[soc] += 1

            raw_skills = parse_skill_list(
                row.get("SKILLS_NAME")
            ) + parse_skill_list(row.get("SOFTWARE_SKILLS_NAME"))
            for sk in set(raw_skills):
                clean_sk = str(sk).strip().title()
                if (
                    len(clean_sk) > 2
                    and clean_sk.lower() not in {"general", "other", "n/a"}
                ):
                    soc_skill_counts[(soc, clean_sk)] += 1

    rows = []
    for (soc, sk), count in soc_skill_counts.items():
        total = soc_total_postings[soc]
        if total > 0:
            rows.append(
                {
                    "SOC": soc,
                    "Skill": sk,
                    "Skill_Postings": count,
                    "Total_Postings": total,
                    "Demand_Rate": count / total,
                    "Metric_Source": "Calculated from df_2016_2025.csv postings",
                }
            )

    lc = pd.DataFrame(rows)
    if lc.empty:
        raise ValueError(
            "No matching 2025 postings found in Lightcast file for the selected occupations."
        )

    selected = universe[universe.Growth_Group != "Middle 50%"].merge(
        lc, on="SOC", how="inner"
    )

    group_rows = []
    for key, g in selected.groupby(["Sector", "Growth_Group", "Skill"]):
        group_rows.append(
            {
                "Sector": key[0],
                "Growth_Group": key[1],
                "Skill": key[2],
                "Mean_SOC_Demand_Rate": g.Demand_Rate.mean(),
                "Employment_Weighted_Demand_Rate": weighted_mean(
                    g.Demand_Rate, g.Sector_Employment
                ),
                "Occupation_Count": g.SOC.nunique(),
                "Skill_Postings": g.Skill_Postings.sum(min_count=1),
            }
        )

    summary = pd.DataFrame(group_rows)
    pivot = summary.pivot_table(
        index=["Sector", "Skill"],
        columns="Growth_Group",
        values="Employment_Weighted_Demand_Rate",
    ).reset_index()
    pivot.columns.name = None
    for c in ["Fast-Growth Quartile", "Slow-Growth Quartile"]:
        if c not in pivot:
            pivot[c] = np.nan

    pivot["Fast_Minus_Slow"] = (
        pivot["Fast-Growth Quartile"] - pivot["Slow-Growth Quartile"]
    )
    pivot["Absolute_Difference"] = pivot.Fast_Minus_Slow.abs()

    top = (
        summary.sort_values(
            "Employment_Weighted_Demand_Rate", ascending=False
        )
        .groupby(["Sector", "Growth_Group"], group_keys=False)
        .head(TOP_N_RESULTS)
    )
    differentiators = (
        pivot[pivot.Fast_Minus_Slow > 0]
        .sort_values("Fast_Minus_Slow", ascending=False)
        .groupby("Sector", group_keys=False)
        .head(TOP_N_RESULTS)
    )

    return selected, summary, top, pivot, differentiators

## 7. CHARTS


In [46]:
def difference_chart(df, sector, label_col, value_col, title, filename, n=15):
    d = df[df.Sector == sector].dropna(subset=[value_col]).nlargest(n, "Absolute_Difference")
    d = d.sort_values(value_col)
    if d.empty:
        return
    colors = [FAST_COLOR if x >= 0 else SLOW_COLOR for x in d[value_col]]
    fig, ax = plt.subplots(figsize=(11, 8))
    ax.barh(d[label_col], d[value_col], color=colors)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(title, weight="bold")
    ax.set_xlabel("Positive = more prominent in fast-growth occupations")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUTPUT_FOLDER / filename, dpi=300, bbox_inches="tight")
    plt.close()


## 8. RUN AND EXPORT


In [48]:
def main():
    require_files()
    staffing = load_staffing()
    projections = load_projections()
    universe = build_occupation_universe(staffing, projections)

    print("\nOccupation counts by sector and growth group:")
    print(universe.groupby(["Sector", "Growth_Group"]).SOC.nunique())

    onet_selected, onet_summary, onet_top, onet_compare, onet_diff = analyze_onet(universe)
    lc_selected, lc_summary, lc_top, lc_compare, lc_diff = analyze_lightcast(universe)

    for sector in SECTOR_NAICS_PREFIXES:
        # FILTER STRICTLY FOR PURE BROAD O*NET SKILLS (IMPORTS & KSA_TYPE == 'Skill')
        od = onet_compare[
            (onet_compare.Sector == sector) & 
            (onet_compare.Scale == "Importance") & 
            (onet_compare.KSA_Type == "Skill")
        ].copy()
        
        od["Label"] = od.KSA_Name
        
        # Generate Clean Broad Skill Chart
        difference_chart(
            od, sector, "Label", "Fast_Minus_Slow",
            f"{sector}: Key O*NET Skill Growth Drivers\n(Fast- vs. Slow-Growth Quartiles)",
            f"{sector}_ONET_Broad_Skills_Differences.png"
        )

    methodology = pd.DataFrame({
        "Component": ["Occupation universe", "Growth groups", "O*NET role", "O*NET scales",
                      "O*NET reporting", "Lightcast role", "Lightcast metric", "Weighting", "Interpretation"],
        "Method": [
            "Detailed SOCs employed in Healthcare NAICS 62, Manufacturing NAICS 31-33, and Semiconductor NAICS 334 using Phoenix-area staffing patterns.",
            "Within each sector, the highest-growth quartile is compared with the lowest-growth quartile using 2024-2034 projected percentage growth.",
            "Measures foundational occupational Knowledge, Skills, and Abilities.",
            "Importance and Level are retained and reported separately; they are never averaged together.",
            "Pure O*NET Skills taxonomy is emphasized to deliver broad, standardized executive competencies.",
            "Measures specific skills currently requested in employer job postings.",
            "Percentage/share of postings associated with each skill within an SOC, taken from Lightcast or calculated from unique raw posting IDs.",
            "SOC-level O*NET and Lightcast measures are weighted by employment within each target sector.",
            "O*NET describes foundational capabilities; Lightcast describes current employer-requested skills.",
        ]})

    output = OUTPUT_FOLDER / "Walmart_Dual_Source_ONET_Lightcast_Analysis.xlsx"
    with pd.ExcelWriter(output, engine="xlsxwriter") as writer:
        methodology.to_excel(writer, "Methodology", index=False)
        universe.to_excel(writer, "Occupation_Universe", index=False)
        onet_top.to_excel(writer, "ONET_Top_by_KSA_Scale", index=False)
        onet_diff.to_excel(writer, "ONET_Fast_Growth_Diffs", index=False)
        onet_compare.to_excel(writer, "ONET_All_Comparisons", index=False)
        onet_summary.to_excel(writer, "ONET_Group_Summary", index=False)
        lc_top.to_excel(writer, "Lightcast_Top_Skills", index=False)
        lc_diff.to_excel(writer, "Lightcast_Fast_Growth", index=False)
        lc_compare.to_excel(writer, "Lightcast_Comparisons", index=False)
        lc_summary.to_excel(writer, "Lightcast_Group_Summary", index=False)
        onet_selected.to_excel(writer, "Underlying_ONET", index=False)
        lc_selected.to_excel(writer, "Underlying_Lightcast", index=False)

        wb = writer.book
        header = wb.add_format({"bold": True, "text_wrap": True, "valign": "top"})
        pct = wb.add_format({"num_format": "0.0%"})
        for name, ws in writer.sheets.items():
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, 0, max(0, ws.dim_colmax))
            ws.set_row(0, None, header)
            ws.set_column(0, ws.dim_colmax, 18)
            if "Lightcast" in name:
                ws.set_column(3, ws.dim_colmax, 18, pct)

    print(f"\nSUCCESS\nWorkbook: {output}\nCharts: {OUTPUT_FOLDER}")

main()

FileNotFoundError: Lightcast folder not found: C:\Users\301533\Documents\Walmart\Lightcast Large Files